# 07 Apply Views — Orchestrator

**Doel:** Achter elkaar uitvoeren van alle view-notebooks in `views/integration/` en
`views/datamart/`. Views kunnen niet binnen DLT bestaan (DLT beheert
Materialised Views en Streaming Tables, geen plain views), dus elke view is een
eigen notebook met één `%sql CREATE OR REPLACE VIEW` cel. Deze orchestrator
roept ze op met `dbutils.notebook.run()` en geeft de `catalog`-widget door.

## Volgorde

1. `integration/sales_line` — eerst, want `fact_sales_line` MV (downstream in
   `dlt_datamart`) leest hiervan.
2. `datamart/dim_*` — 9 dim-notebooks, geen onderlinge afhankelijkheden.

```
setup
 ├─→ ingest_full
 └─→ ingest_incremental
       ↓
       dlt_integration                  (Streaming Tables in INTEGRATION)
        ↓
        apply_views                     (deze notebook — fan-out naar 10 view-notebooks)
         ↓
         dlt_datamart                   (fact_order MV + fact_sales_line MV — leest sales_line view)
```

## Patroon-overzicht in de stack

| Object | Patroon | Reden |
|---|---|---|
| `STAGING_AZURESTORAGE.*` | Delta Table | Landing zone, CDF, Time Travel |
| `INTEGRATION.order_*` + quarantines | Streaming Table | apply_changes state, MERGE |
| `INTEGRATION.sales_line` | **View** (eigen notebook) | Pure join, geen state, altijd vers |
| `DATAMART.dim_*` (9 stuks) | **View** (eigen notebook per dim) | Lage cardinaliteit, simpele projecties |
| `DATAMART.fact_order` | MV (in `dlt_datamart`) | Order-grain fact, materialised zodat Silver-correcties automatisch propageren |
| `DATAMART.fact_sales_line` | MV (Liquid Clustering, in `dlt_datamart`) | Zwaarste tabel, profiteert van clustering |

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Catalog: {catalog}")

In [ ]:
# Volgorde — sales_line eerst (fact_sales_line MV in dlt_datamart leest hiervan),
# daarna de 9 dim-notebooks. Paden zijn relatief t.o.v. deze notebook.
view_notebooks = [
    "integration/sales_line",
    "datamart/dim_date",
    "datamart/dim_truck",
    "datamart/dim_location",
    "datamart/dim_customer",
    "datamart/dim_menu_item",
    "datamart/dim_currency",
    "datamart/dim_order_channel",
    "datamart/dim_shift",
    "datamart/dim_discount",
]

for nb in view_notebooks:
    dbutils.notebook.run(nb, 0, {"catalog": catalog})
    print(f"  applied: {nb}")

## Validatie

Lijst alle views in `INTEGRATION` + `DATAMART` om te bevestigen dat alles
aangemaakt is.

In [ ]:
for schema in ("INTEGRATION", "DATAMART"):
    rows = spark.sql(f"SHOW VIEWS IN {catalog}.{schema}").collect()
    print(f"\n{catalog}.{schema} — {len(rows)} view(s):")
    for r in rows:
        print(f"  {r['viewName']}")